In [ ]:
import truststore

import planetary_computer
import pystac_client

from satellite_imagery_viewer.models.imagery import ImagerySearchRequest, ImagerySearchResponse
from satellite_imagery_viewer.services.discovery import discover_imagery


# ------------------------------------------------------------------
# ENVIRONMENT SETUP
# ------------------------------------------------------------------

# Uses the operating system certificate store.
# Required on the work PC because of corporate SSL certificate handling.
# Harmless on the personal PC.
truststore.inject_into_ssl()


# ------------------------------------------------------------------
# PLANETARY COMPUTER STAC CLIENT
# ------------------------------------------------------------------

catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)


# ------------------------------------------------------------------
# COMMON TEST SEARCH INPUTS
# ------------------------------------------------------------------

area_of_interest = {
    "type": "Polygon",
    "coordinates": [
        [
            [-122.522, 37.7045],
            [-122.356, 37.7045],
            [-122.356, 37.815],
            [-122.522, 37.815],
            [-122.522, 37.7045],
        ]
    ],
}

time_of_interest = "2023-01-01/2023-12-31"

collection_ids = [
    "sentinel-2-l2a",
    "landsat-c2-l2",
    "sentinel-1-grd",
]

print(ImagerySearchResponse.model_fields.keys())

# ------------------------------------------------------------------
# END-TO-END DISCOVERY SMOKE TEST
# ------------------------------------------------------------------

for collection_id in collection_ids:
    search_params = ImagerySearchRequest(
        collection=collection_id,
        area_of_interest=area_of_interest,
        time_range=time_of_interest,
    )

    results = discover_imagery(search_params)

    print(f"\n{collection_id}")
    print(f"Result count: {len(results)}")

    for result in results[:3]:
        print(result)


sentinel-2-l2a
Result count: 125
id='S2B_MSIL2A_20231230T185809_R113_T10SEG_20231230T223401' datetime='2023-12-30T18:58:09.024000Z' platform='Sentinel-2B' collection='sentinel-2-l2a' geometry=PolygonGeometry(type='Polygon', coordinates=[[(-123.0002276, 37.9475896), (-121.7506457, 37.9409562), (-121.7669948, 36.9514795), (-123.0002247, 36.9578811), (-123.0002276, 37.9475896)]]) proj_code='EPSG:32610' cloud_cover=44.053599
id='S2A_MSIL2A_20231225T185811_R113_T10SEG_20231225T231003' datetime='2023-12-25T18:58:11.024000Z' platform='Sentinel-2A' collection='sentinel-2-l2a' geometry=PolygonGeometry(type='Polygon', coordinates=[[(-123.0002276, 37.9475896), (-121.7506457, 37.9409562), (-121.7669948, 36.9514795), (-123.0002247, 36.9578811), (-123.0002276, 37.9475896)]]) proj_code='EPSG:32610' cloud_cover=98.834991
id='S2B_MSIL2A_20231220T185809_R113_T10SEG_20231222T025826' datetime='2023-12-20T18:58:09.024000Z' platform='Sentinel-2B' collection='sentinel-2-l2a' geometry=PolygonGeometry(type='P

In [17]:
from pprint import pprint


COLLECTION_IDS = [
    "sentinel-2-l2a",
    "landsat-c2-l2",
    "sentinel-1-grd",
]

SAMPLE_SIZE = 5


def get_queryable_info(collection):
    queryables = collection.get_queryables()
    properties = queryables.get("properties", {})
    summaries = collection.summaries.to_dict()

    result = {}

    for field_name, schema in properties.items():
        result[field_name] = {
            "type": schema.get("type"),
            "title": schema.get("title"),
            "description": schema.get("description"),
            "enum": schema.get("enum"),
            "summary": summaries.get(field_name),
        }

    return result


def get_sample_items(collection_id):
    search = catalog.search(
        collections=[collection_id],
        intersects=area_of_interest,
        datetime=time_of_interest,
        max_items=SAMPLE_SIZE,
    )

    return list(search.items())


def get_keys_present_in_every_item(items, key_source):
    if not items:
        return set()

    key_sets = []

    for item in items:
        if key_source == "top_level":
            keys = set(item.to_dict().keys())
        elif key_source == "properties":
            keys = set(item.properties.keys())
        else:
            raise ValueError(f"Unknown key source: {key_source}")

        key_sets.append(keys)

    return set.intersection(*key_sets)


def get_sample_property_values(items, property_names):
    values = {}

    for property_name in property_names:
        values[property_name] = []

        for item in items:
            value = item.properties.get(property_name)

            if value not in values[property_name]:
                values[property_name].append(value)

    return values


# ------------------------------------------------------------
# 1. QUERYABLE RESEARCH
# ------------------------------------------------------------

queryable_info_by_collection = {}

for collection_id in COLLECTION_IDS:
    collection = catalog.get_collection(collection_id)
    queryable_info_by_collection[collection_id] = get_queryable_info(collection)


queryable_sets = {
    collection_id: set(fields.keys())
    for collection_id, fields in queryable_info_by_collection.items()
}

common_queryables = set.intersection(*queryable_sets.values())


print("=" * 80)
print("EXACT QUERYABLES COMMON TO ALL COLLECTIONS")
print("=" * 80)

for field in sorted(common_queryables):
    print(field)


print("\n" + "=" * 80)
print("QUERYABLE DETAILS FOR COMMON FIELDS")
print("=" * 80)

for field in sorted(common_queryables):
    print(f"\nFIELD: {field}")

    for collection_id in COLLECTION_IDS:
        info = queryable_info_by_collection[collection_id][field]

        print(f"\n  {collection_id}")
        print(f"    type:        {info['type']}")
        print(f"    title:       {info['title']}")
        print(f"    description: {info['description']}")
        print(f"    enum:        {info['enum']}")
        print(f"    summary:     {info['summary']}")


print("\n" + "=" * 80)
print("COLLECTION-SPECIFIC QUERYABLES")
print("=" * 80)

for collection_id in COLLECTION_IDS:
    collection_specific = (
        queryable_sets[collection_id] - common_queryables
    )

    print(f"\n{collection_id}")

    for field in sorted(collection_specific):
        info = queryable_info_by_collection[collection_id][field]

        print(
            f"  {field} | "
            f"type={info['type']} | "
            f"summary={info['summary']}"
        )


# ------------------------------------------------------------
# 2. RESPONSE / ITEM METADATA RESEARCH
# ------------------------------------------------------------

items_by_collection = {}

for collection_id in COLLECTION_IDS:
    items = get_sample_items(collection_id)
    items_by_collection[collection_id] = items

    print(
        f"\nFetched {len(items)} sample items "
        f"for {collection_id}"
    )


stable_top_level_keys = {}
stable_property_keys = {}

for collection_id, items in items_by_collection.items():
    stable_top_level_keys[collection_id] = (
        get_keys_present_in_every_item(items, "top_level")
    )

    stable_property_keys[collection_id] = (
        get_keys_present_in_every_item(items, "properties")
    )


common_top_level_keys = set.intersection(
    *stable_top_level_keys.values()
)

common_property_keys = set.intersection(
    *stable_property_keys.values()
)


print("\n" + "=" * 80)
print("TOP-LEVEL STAC FIELDS COMMON TO ALL THREE")
print("=" * 80)

for field in sorted(common_top_level_keys):
    print(field)


print("\n" + "=" * 80)
print("ITEM PROPERTY FIELDS COMMON TO ALL THREE")
print("=" * 80)

for field in sorted(common_property_keys):
    print(field)


# ------------------------------------------------------------
# 3. SAMPLE VALUES FOR COMMON RESPONSE PROPERTIES
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("SAMPLE VALUES FOR COMMON ITEM PROPERTIES")
print("=" * 80)

for field in sorted(common_property_keys):
    print(f"\nFIELD: {field}")

    for collection_id in COLLECTION_IDS:
        values = get_sample_property_values(
            items_by_collection[collection_id],
            [field],
        )[field]

        print(f"  {collection_id}: {values}")


# ------------------------------------------------------------
# 4. STABLE BUT COLLECTION-SPECIFIC RESPONSE PROPERTIES
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("STABLE COLLECTION-SPECIFIC ITEM PROPERTIES")
print("=" * 80)

for collection_id in COLLECTION_IDS:
    collection_specific = (
        stable_property_keys[collection_id]
        - common_property_keys
    )

    print(f"\n{collection_id}")

    for field in sorted(collection_specific):
        values = get_sample_property_values(
            items_by_collection[collection_id],
            [field],
        )[field]

        print(f"  {field}: {values}")

EXACT QUERYABLES COMMON TO ALL COLLECTIONS
datetime
end_datetime
geometry
id
start_datetime

QUERYABLE DETAILS FOR COMMON FIELDS

FIELD: datetime

  sentinel-2-l2a
    type:        string
    title:       Acquired
    description: Datetime
    enum:        None
    summary:     None

  landsat-c2-l2
    type:        string
    title:       Acquired
    description: Datetime
    enum:        None
    summary:     None

  sentinel-1-grd
    type:        string
    title:       Acquired
    description: Datetime
    enum:        None
    summary:     None

FIELD: end_datetime

  sentinel-2-l2a
    type:        string
    title:       End datetime
    description: End datetime
    enum:        None
    summary:     None

  landsat-c2-l2
    type:        string
    title:       End datetime
    description: End datetime
    enum:        None
    summary:     None

  sentinel-1-grd
    type:        string
    title:       End datetime
    description: End datetime
    enum:        None
    s